In [1]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [2]:
output_dir = 'outputs_new'

output_dir = Path('/data/vision/polina/users/marcusbl/bin_class') / output_dir

model_name = 'model_loss'
metrics = ['auc', 'acc']

Parse results into Pandas DF

In [3]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [4]:
df['model_auc']

acc    auc  epoch     f1  fn  fp    fpr   loss   prec  recall  \
test  run                                                                     
test1 run0  0.876  0.924     -1  0.684  36  61  0.095  0.277  0.633   0.745   
      run1  0.909  0.952     -1  0.729  21  51  0.075  0.222  0.655   0.822   
      run2  0.910  0.954     -1  0.750  47  27  0.041  0.243  0.804   0.703   
      run3  0.849  0.930     -1  0.699  23  88  0.151  0.328  0.594   0.849   
      run4  0.931  0.947     -1  0.787  41  11  0.018  0.209  0.897   0.701   

             tn   tp    tpr  
test  run                    
test1 run0  578  105  0.745  
      run1  625   97  0.822  
      run2  635  111  0.703  
      run3  493  129  0.849  
      run4  609   96  0.701

Average/Variance for desired metrics over all runs for the desired model

In [5]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc              acc          
        mean      var    mean       var
test                                   
test1  0.932  0.00047  0.8916  0.000758